# Problem Set III - Question 2: Classical to Quantum Circuit Conversion

**Course:** QPROG - Quantum Programming  
**Students:** Jady Pâmella Barbacena da Silva and Navyashree Suryanarayana Rao Prasanna
**Group:** Group ProblemSet-III-N 
**Assignment:** Converting Classical Circuits to Quantum Circuits  

This notebook implements the conversion of classical circuits (AND, OR, XOR, NOT, NAND gates) into quantum circuits.

## Environment Setup

Install required packages and import the ClassicalCircuit class.

In [1]:
%pip install qiskit==2.2.3
%pip install qiskit-aer==0.17.2
%pip install matplotlib

In [2]:
# Import required libraries
from qiskit import QuantumCircuit

## ClassicalCircuit Class

Define the ClassicalCircuit class that reads and converts classical circuits to quantum circuits.

In [3]:
class ClassicalCircuit:
    def __init__(self, filename):
        self.n_inputs = 0
        self.n_outputs = 0
        self.n_internal = 0
        self.input_gates = []
        self.output_gates = []
        self.internal_gates = []
        self.gates = []
        self.read(filename)
    
    def read(self, filename):
        """Read circuit definition from file."""
        with open(filename, 'r') as file:
            lines = file.readlines()
        
        self.n_inputs = int(lines[0].strip())
        self.n_outputs = int(lines[1].strip())
        self.n_internal = int(lines[2].strip())
        self.input_gates = list(map(int, lines[3].strip().split()))
        self.output_gates = list(map(int, lines[4].strip().split()))
        self.internal_gates = list(map(int, lines[5].strip().split()))
        
        for line in lines[6:]:
            gate = line.strip().split()
            gate[0] = int(gate[0])
            if gate[1] in ["and", "or", "xor", "nand"]:
                gate[2] = int(gate[2])
                gate[3] = int(gate[3])
            elif gate[1] == "not":
                gate[2] = int(gate[2])
            self.gates.append(gate)
    
    def print(self):
        """Print circuit information."""
        print(f"n_inputs: {self.n_inputs}")
        print(f"n_outputs: {self.n_outputs}")
        print(f"n_internal: {self.n_internal}")
        print(f"Input Gates: {self.input_gates}")
        print(f"Output Gates: {self.output_gates}")
        print(f"Internal Gates: {self.internal_gates}")
        print("Gates:")
        for gate in self.gates:
            print(gate)
        print()

## Task 2.3: Implement convert_step_1

Convert classical gates using ancilla qubits initialized to |0⟩.

In [4]:
def convert_step_1(self, quantumCircuit):
    """
    Convert classical gates to quantum gates using ancilla qubits at |0⟩.
    """
    for gate in self.gates:
        output = gate[0]
        gate_type = gate[1]
        
        if gate_type == "not":
            input1 = gate[2]
            quantumCircuit.cx(input1, output)
            quantumCircuit.x(output)
        
        elif gate_type == "and":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.ccx(input1, input2, output)
        
        elif gate_type == "or":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.x(input1)
            quantumCircuit.x(input2)
            quantumCircuit.ccx(input1, input2, output)
            quantumCircuit.x(input1)
            quantumCircuit.x(input2)
            quantumCircuit.x(output)
        
        elif gate_type == "xor":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.cx(input1, output)
            quantumCircuit.cx(input2, output)
        
        elif gate_type == "nand":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.ccx(input1, input2, output)
            quantumCircuit.x(output)

# Add method to ClassicalCircuit class
ClassicalCircuit.convert_step_1 = convert_step_1

## Task 2.4: Implement convert_step_2

Convert and uncompute intermediate qubits to clean up ancilla.

In [5]:
def convert_step_2(self, quantumCircuit):
    """
    Convert classical gates and uncompute internal qubits.
    """
    # First pass: compute all gates
    for gate in self.gates:
        output = gate[0]
        gate_type = gate[1]
        
        if gate_type == "not":
            input1 = gate[2]
            quantumCircuit.cx(input1, output)
            quantumCircuit.x(output)
        
        elif gate_type == "and":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.ccx(input1, input2, output)
        
        elif gate_type == "or":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.x(input1)
            quantumCircuit.x(input2)
            quantumCircuit.ccx(input1, input2, output)
            quantumCircuit.x(input1)
            quantumCircuit.x(input2)
            quantumCircuit.x(output)
        
        elif gate_type == "xor":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.cx(input1, output)
            quantumCircuit.cx(input2, output)
        
        elif gate_type == "nand":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.ccx(input1, input2, output)
            quantumCircuit.x(output)
    
    # Uncompute internal qubits (reverse order)
    for gate in reversed(self.gates):
        output = gate[0]
        if output in self.internal_gates:
            gate_type = gate[1]
            
            if gate_type == "not":
                input1 = gate[2]
                quantumCircuit.x(output)
                quantumCircuit.cx(input1, output)
            
            elif gate_type == "and":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.ccx(input1, input2, output)
            
            elif gate_type == "or":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.x(output)
                quantumCircuit.x(input1)
                quantumCircuit.x(input2)
                quantumCircuit.ccx(input1, input2, output)
                quantumCircuit.x(input1)
                quantumCircuit.x(input2)
            
            elif gate_type == "xor":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.cx(input2, output)
                quantumCircuit.cx(input1, output)
            
            elif gate_type == "nand":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.x(output)
                quantumCircuit.ccx(input1, input2, output)

# Add method to ClassicalCircuit class
ClassicalCircuit.convert_step_2 = convert_step_2

## Task 2.5 & 2.6: Implement convert

Complete conversion with ancilla duplication for output qubits.

In [6]:
def convert(self, quantumCircuit):
    """
    Complete conversion with ancilla duplication for output preservation.
    Circuit has n_inputs + 2*n_outputs + n_internal qubits.
    """
    # Compute all gates
    for gate in self.gates:
        output = gate[0]
        gate_type = gate[1]
        
        if gate_type == "not":
            input1 = gate[2]
            quantumCircuit.cx(input1, output)
            quantumCircuit.x(output)
        
        elif gate_type == "and":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.ccx(input1, input2, output)
        
        elif gate_type == "or":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.x(input1)
            quantumCircuit.x(input2)
            quantumCircuit.ccx(input1, input2, output)
            quantumCircuit.x(input1)
            quantumCircuit.x(input2)
            quantumCircuit.x(output)
        
        elif gate_type == "xor":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.cx(input1, output)
            quantumCircuit.cx(input2, output)
        
        elif gate_type == "nand":
            input1 = gate[2]
            input2 = gate[3]
            quantumCircuit.ccx(input1, input2, output)
            quantumCircuit.x(output)
    
    # Copy output qubits to ancilla qubits
    ancilla_offset = self.n_inputs + self.n_outputs + self.n_internal
    for i, output_gate in enumerate(self.output_gates):
        quantumCircuit.cx(output_gate, ancilla_offset + i)
    
    # Uncompute internal qubits
    for gate in reversed(self.gates):
        output = gate[0]
        if output in self.internal_gates:
            gate_type = gate[1]
            
            if gate_type == "not":
                input1 = gate[2]
                quantumCircuit.x(output)
                quantumCircuit.cx(input1, output)
            
            elif gate_type == "and":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.ccx(input1, input2, output)
            
            elif gate_type == "or":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.x(output)
                quantumCircuit.x(input1)
                quantumCircuit.x(input2)
                quantumCircuit.ccx(input1, input2, output)
                quantumCircuit.x(input1)
                quantumCircuit.x(input2)
            
            elif gate_type == "xor":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.cx(input2, output)
                quantumCircuit.cx(input1, output)
            
            elif gate_type == "nand":
                input1 = gate[2]
                input2 = gate[3]
                quantumCircuit.x(output)
                quantumCircuit.ccx(input1, input2, output)

# Add method to ClassicalCircuit class
ClassicalCircuit.convert = convert

## Test the Implementation

Test all three conversion methods with the provided circuit.txt file.

In [7]:
# Load and print circuit
cc = ClassicalCircuit("circuit.txt")
cc.print()

n_inputs: 3
n_outputs: 2
n_internal: 2
Input Gates: [0, 1, 2]
Output Gates: [3, 4]
Internal Gates: [5, 6]
Gates:
[5, 'and', 0, 1]
[3, 'not', 5]
[6, 'not', 2]
[4, 'and', 5, 6]



In [8]:
# Test convert_step_1
n_wires = cc.n_inputs + cc.n_outputs + cc.n_internal
qc1 = QuantumCircuit(n_wires, 0)
cc.convert_step_1(qc1)
print("Convert Step 1:")
print(qc1)
print()

Convert Step 1:
                         
q_0: ──■─────────────────
       │                 
q_1: ──■─────────────────
       │                 
q_2: ──┼────■────────────
       │    │  ┌───┐┌───┐
q_3: ──┼────┼──┤ X ├┤ X ├
       │    │  └─┬─┘├───┤
q_4: ──┼────┼────┼──┤ X ├
     ┌─┴─┐  │    │  └─┬─┘
q_5: ┤ X ├──┼────■────■──
     └───┘┌─┴─┐┌───┐  │  
q_6: ─────┤ X ├┤ X ├──■──
          └───┘└───┘     



In [9]:
# Test convert_step_2
n_wires = cc.n_inputs + cc.n_outputs + cc.n_internal
qc2 = QuantumCircuit(n_wires, 0)
cc.convert_step_2(qc2)
print("Convert Step 2:")
print(qc2)
print()

Convert Step 2:
                                   
q_0: ──■───────────────────■───────
       │                   │       
q_1: ──■───────────────────■───────
       │                   │       
q_2: ──┼────■──────────────┼────■──
       │    │  ┌───┐┌───┐  │    │  
q_3: ──┼────┼──┤ X ├┤ X ├──┼────┼──
       │    │  └─┬─┘├───┤  │    │  
q_4: ──┼────┼────┼──┤ X ├──┼────┼──
     ┌─┴─┐  │    │  └─┬─┘┌─┴─┐  │  
q_5: ┤ X ├──┼────■────■──┤ X ├──┼──
     └───┘┌─┴─┐┌───┐  │  ├───┤┌─┴─┐
q_6: ─────┤ X ├┤ X ├──■──┤ X ├┤ X ├
          └───┘└───┘     └───┘└───┘



In [10]:
# Test convert (complete)
n_wires = cc.n_inputs + 2*cc.n_outputs + cc.n_internal
qc3 = QuantumCircuit(n_wires, 0)
cc.convert(qc3)
print("Complete Convert:")
print(qc3)
print()

Complete Convert:
                                             
q_0: ──■─────────────────────────────■───────
       │                             │       
q_1: ──■─────────────────────────────■───────
       │                             │       
q_2: ──┼────■────────────────────────┼────■──
       │    │  ┌───┐┌───┐            │    │  
q_3: ──┼────┼──┤ X ├┤ X ├──■─────────┼────┼──
       │    │  └─┬─┘├───┤  │         │    │  
q_4: ──┼────┼────┼──┤ X ├──┼────■────┼────┼──
     ┌─┴─┐  │    │  └─┬─┘  │    │  ┌─┴─┐  │  
q_5: ┤ X ├──┼────■────■────┼────┼──┤ X ├──┼──
     └───┘┌─┴─┐┌───┐  │    │    │  ├───┤┌─┴─┐
q_6: ─────┤ X ├┤ X ├──■────┼────┼──┤ X ├┤ X ├
          └───┘└───┘     ┌─┴─┐  │  └───┘└───┘
q_7: ────────────────────┤ X ├──┼────────────
                         └───┘┌─┴─┐          
q_8: ─────────────────────────┤ X ├──────────
                              └───┘          

